In [1]:
import os
import time
import pandas as pd
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [2]:
spark = (
    SparkSession.builder
        .appName("CryptoETL")
        .config("spark.master", "spark://spark-master:7077")
        # ---- Iceberg + Hive Catalog ----
        .config("spark.sql.catalog.hive_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.hive_catalog.catalog-impl", "org.apache.iceberg.hive.HiveCatalog")
        .config("spark.sql.catalog.hive_catalog.uri", "thrift://hive-metastore:9083")
        .config("spark.sql.catalog.hive_catalog.warehouse", "s3a://crypto-data-lake/")
        # ---- Default catalog
        .config("spark.sql.defaultCatalog", "hive_catalog")
        # ---- S3 (MinIO) ----
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        # ---- Iceberg Extensions ----
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        # ---- Extra JARs ----
        .config("spark.jars", ",".join([
            "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
            "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
            "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
        ]))
        .getOrCreate()
)

25/09/28 12:17:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
bucket = "crypto-data-lake"

In [14]:
output_path = f"s3a://{bucket}/landing_zone/spot/daily/aggTrades/BTCUSDT/2025_08"
df = spark.read.parquet(output_path)

In [15]:
df.show()

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------+
|agg_trade_id|    price|quantity|first_trade_id|last_trade_id|       timestamp|is_buyer_maker|is_best_match|ingest_date|    ingest_timestamp|
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------+
|  3657365721|117095.22| 0.00418|    5180690854|   5180690854|1755894576003465|          true|         true| 2025-09-28|2025-09-28 12:05:...|
|  3657365722|117095.23| 0.00379|    5180690855|   5180690855|1755894576285197|         false|         true| 2025-09-28|2025-09-28 12:05:...|
|  3657365723|117095.23| 0.00113|    5180690856|   5180690867|1755894576470523|         false|         true| 2025-09-28|2025-09-28 12:05:...|
|  3657365724|117095.23| 0.05998|    5180690868|   5180690889|1755894576515890|         false|         true| 2025-09-28|2025-09-28 12:05:...|
|  365

In [16]:
df.count()

24302731

In [17]:
# .orderBy("timestamp", ascending=True) \
# .sample(0.001) \
# .select(["agg_trade_id", "timestamp", "timestamp_date", "timestamp_second", "group_id", "group_date"]) \
# .show(truncate=False)
df = df.withColumn("timestamp_date", F.from_unixtime(F.col("timestamp") / 1_000_000)) \
    .withColumn("timestamp_second", (F.col("timestamp") / 1_000_000).cast("long")) \
    .withColumn("group_id", (F.col("timestamp_second") / 900).cast("long")) \
    .withColumn("group_date", F.from_unixtime(F.col("group_id") * 900)) \
    .withColumn("transform_date", F.current_date()) \
    .withColumn("transform_timestamp", F.current_timestamp()).orderBy("timestamp")

In [18]:
df.writeTo("transform_db.aggtrades_month").tableProperty("format-version", "2").createOrReplace()

In [19]:
spark.sql("""
select count(*) from transform_db.aggtrades_month
""").show()

+--------+
|count(1)|
+--------+
|24302731|
+--------+



In [9]:
spark.sql("""
select 
    group_id,
    group_date,
    max(price) as high_price,
    min(price) as low_price,
    max(agg_trade_id) as max_id,
    min(agg_trade_id) as min_id,
    sum(quantity) as volume
from 
    transform_db.aggtrades_month
group by group_id, group_date
""").show()

[Stage 6:===============================================>           (4 + 1) / 5]

+--------+-------------------+----------+---------+----------+----------+------------------+
|group_id|         group_date|high_price|low_price|    max_id|    min_id|            volume|
+--------+-------------------+----------+---------+----------+----------+------------------+
| 1951112|2025-08-24 02:00:00| 115666.68|115419.54|3658007824|3658002316| 92.74401000000165|
| 1949105|2025-08-03 04:15:00| 113530.25|113407.91|3642583783|3642580414| 59.36983000000011|
| 1949148|2025-08-03 15:00:00| 113792.29| 113680.0|3642752702|3642748182|44.461900000000135|
| 1949250|2025-08-04 16:30:00| 115161.21| 114900.0|3643317349|3643309516|158.20397000000233|
| 1949362|2025-08-05 20:30:00| 113989.91|113624.01|3644130202|3644125424| 75.53828000000115|
| 1949450|2025-08-06 18:30:00|  115573.2|115301.27|3644682092|3644677007| 92.11517000000164|
| 1951003|2025-08-22 22:45:00| 116776.08|116664.27|3657430731|3657426489|101.26393000000047|
| 1951175|2025-08-24 17:45:00| 114555.29|114308.34|3658359825|36583496

In [10]:
spark.sql("""
select 
    agg_trade_id,
    price,
    quantity,
    timestamp
from 
    transform_db.aggtrades_month
""").show()

+------------+---------+--------+----------------+
|agg_trade_id|    price|quantity|       timestamp|
+------------+---------+--------+----------------+
|  3657365721|117095.22| 0.00418|1755894576003465|
|  3657365722|117095.23| 0.00379|1755894576285197|
|  3657365723|117095.23| 0.00113|1755894576470523|
|  3657365724|117095.23| 0.05998|1755894576515890|
|  3657365725|117095.24|  7.5E-4|1755894576515890|
|  3657365726|117096.13| 0.00183|1755894576515890|
|  3657365727|117096.53|  5.0E-5|1755894576515890|
|  3657365728|117097.35| 0.00137|1755894576515890|
|  3657365729|117097.48|  1.0E-4|1755894576515890|
|  3657365730| 117097.5| 0.00161|1755894576515890|
|  3657365731|117097.53|  5.0E-5|1755894576515890|
|  3657365732| 117097.7|  5.0E-5|1755894576515890|
|  3657365733|117097.94|  2.1E-4|1755894576515890|
|  3657365734|117098.71|  7.6E-4|1755894576515890|
|  3657365735|117099.45|  5.0E-5|1755894576515890|
|  3657365736|117099.46|  1.0E-4|1755894576515890|
|  3657365737|117099.99|   0.00

In [20]:
df_kline = spark.sql("""
select 
    group_id,
    group_date,
    round(first(timestamp, true), 2) as open_time,
    round(first(price, true), 2) as open_price,
    round(max(price), 2) as high_price,
    round(min(price), 2) as low_price,
    round(last(price, true), 2) as close_price,
    round(sum(quantity), 2) as volume,
    last(timestamp, true) as close_time
from transform_db.aggtrades_month
group by group_id, group_date
order by group_id
""")

In [21]:
df_kline.show()

[Stage 33:===================================================>    (11 + 1) / 12]

+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price| volume|      close_time|
+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+
| 1948896|2025-08-01 00:00:00|1754006400328945| 115764.07| 115829.46|115308.55|  115313.01| 302.16|1754007299467573|
| 1948897|2025-08-01 00:15:00|1754007300010950| 115313.01|  115933.0| 115313.0|  115800.01| 450.54|1754008199447993|
| 1948898|2025-08-01 00:30:00|1754008200077603|  115800.0|  115800.0|115423.87|  115517.98| 184.43|1754009099900832|
| 1948899|2025-08-01 00:45:00|1754009100223687| 115517.99| 115527.53|114313.13|  115427.27|1589.76|1754009999974074|
| 1948900|2025-08-01 01:00:00|1754010000363342| 115427.27| 115609.99| 114600.0|   114649.9| 681.88|1754010899995166|
| 1948901|2025-08-01 01:15:00|1754010900041356|  114649.9| 11527

In [22]:
df_kline.writeTo("serving_db.klines_month").tableProperty("format-version", "2").createOrReplace()

In [23]:
df_kline.show()

[Stage 44:==============================================>         (10 + 2) / 12]

+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price| volume|      close_time|
+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+
| 1948896|2025-08-01 00:00:00|1754006400328945| 115764.07| 115829.46|115308.55|  115313.01| 302.16|1754007299467573|
| 1948897|2025-08-01 00:15:00|1754007300010950| 115313.01|  115933.0| 115313.0|  115800.01| 450.54|1754008199447993|
| 1948898|2025-08-01 00:30:00|1754008200077603|  115800.0|  115800.0|115423.87|  115517.98| 184.43|1754009099900832|
| 1948899|2025-08-01 00:45:00|1754009100223687| 115517.99| 115527.53|114313.13|  115427.27|1589.76|1754009999974074|
| 1948900|2025-08-01 01:00:00|1754010000363342| 115427.27| 115609.99| 114600.0|   114649.9| 681.88|1754010899995166|
| 1948901|2025-08-01 01:15:00|1754010900041356|  114649.9| 11527

In [3]:
df_sorted = (
    spark.sql("SELECT * FROM serving_db.sma7")
    .coalesce(1) # one partition, not shuffle
    .sortWithinPartitions("group_id")
)

25/09/25 10:34:47 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
schema = types.StructType([
    *df_sorted.schema.fields,  # keep all original fields
    types.StructField("ema7", types.DoubleType(), True)
])

In [5]:
from decimal import Decimal, getcontext, ROUND_HALF_UP

# set precision high enough for finance data
getcontext().prec = 28  

def ema_in_chunks(iterator):  # one stream iterator per partition
    alpha = Decimal(2) / Decimal(7 + 1)  # keep alpha as Decimal
    prev = None
    for pdf in iterator:  # 10,000 rows pandas dataframe for a chunk
        ema = []
        for price in pdf["close_price"]:
            price_dec = Decimal(str(price))  # convert to Decimal exactly
            if prev is None:
                prev = Decimal(str(pdf["ma7"].iloc[0]))  # initialize with SMA7
            else:
                prev = alpha * price_dec + (Decimal(1) - alpha) * prev
            # emulate Spark's round(..., 2)
            ema.append(float(prev.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)))
        pdf["ema7"] = ema
        pdf = pdf[[*pdf.columns[:-1], "ema7"]]
        yield pdf

In [6]:
ema_df = df_sorted.mapInPandas(ema_in_chunks, schema)

In [8]:
ema_df.writeTo("serving_db.ema7").tableProperty("format-version", "2").createOrReplace()